# 라벨 경계: 오답이 어디에 몰려 있나

v4 3분류 기준선의 오답을 라벨별로 갈라 보고, 두 소수 클래스를 합친 2분류와 비교합니다.
fold 평균이 아니라 924건 통합 OOF로 계산하므로 `v4_baseline_results.md`의 fold 평균값과는
집계 방식이 다릅니다.

읽고 나면 답이 되는 질문 — 3분류가 어려운 것이 **모델의 한계인가, 과제 정의의 난이도인가.**

재현 명령:

```
$env:RFP_DATASET_VERSION='v4'; python -m scripts.evaluation.binary_review
$env:RFP_DATASET_VERSION='v4'; python -m scripts.evaluation.boundary_cases
```

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager
from sklearn.metrics import precision_recall_fscore_support

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'scripts').is_dir()), Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
installed = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'DejaVu Sans'] if f in installed), 'DejaVu Sans')
plt.rcParams['axes.unicode_minus'] = False

LABELS = ['통상수용', '견적반영', '계약·질의검토']
V4 = ROOT / 'reports/current/v4'
oof = pd.read_csv(V4 / 'model_candidate_oof.csv', encoding='utf-8-sig')
binary = json.loads((V4 / 'binary_review_results.json').read_text(encoding='utf-8'))
boundary = pd.read_csv(V4 / 'boundary_cases.csv', encoding='utf-8-sig')
print(f'3분류 OOF {len(oof)}건 · 2분류 결과 {len(binary["results"])}종 · 경계 사례 {len(boundary)}건')

In [ ]:
pred = oof['word_char_logistic_pred']
p, r, f1, support = precision_recall_fscore_support(oof['gold'], pred, labels=LABELS, zero_division=0)
per_label = pd.DataFrame({'건수': support, 'precision': p, 'recall': r, 'F1': f1}, index=LABELS).round(3)
confusion = pd.crosstab(oof['gold'], pred).reindex(index=LABELS, columns=LABELS, fill_value=0)
confusion.index.name, confusion.columns.name = '정답', '예측'
display(per_label, confusion)
swap = confusion.loc['견적반영', '계약·질의검토'] + confusion.loc['계약·질의검토', '견적반영']
print(f'오답 {(oof["gold"] != pred).sum()}건 중 견적↔계약 상호 혼동 {swap}건')

In [ ]:
trained = {item['name']: item for item in binary['results']}['word 1-2 + char 3-4gram + balanced']
comparison = pd.DataFrame(
    [
        {'설정': '3분류 (통합 OOF)', 'macro F1': f1.mean(), '정확도': (oof['gold'] == pred).mean()},
        {'설정': '2분류 정식 학습', 'macro F1': trained['pooled']['macro_f1'], '정확도': trained['pooled']['accuracy']},
        {'설정': '2분류 사후 접기', 'macro F1': binary['collapsed_three_class_reference']['macro_f1'], '정확도': binary['collapsed_three_class_reference']['accuracy']},
    ]
).set_index('설정').round(4)
display(comparison)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
comparison['macro F1'].plot.bar(ax=axes[0], color=['#e76f51', '#2a9d8f', '#457b9d'], rot=12)
axes[0].set_ylim(0, 1); axes[0].set_title('같은 입력·같은 분할, 라벨만 다르게')
pd.DataFrame({item['name']: [fold['macro_f1'] for fold in item['folds']] for item in binary['results'] if item['name'] != 'Dummy(최빈)'}).plot.box(ax=axes[1], rot=20)
axes[1].set_title('2분류 fold별 macro F1 (10 fold)'); plt.tight_layout()

In [ ]:
from scripts.labeling.label_dataset import load_label_dataset
from scripts.evaluation.folds import evaluation_excluded_uids

rows = [r for r in load_label_dataset(version='v4')[0] if r['requirement_uid'] not in evaluation_excluded_uids()]
has_both = lambda r: bool(r.get('blockers')) and r.get('cost_basis') not in (None, '', '없음')
minority = [r for r in rows if r['primary_action'] != '통상수용']
overlap = pd.Series({
    '혼동 98건': (boundary['blockers'].fillna('').str.strip().ne('') & boundary['cost_basis'].fillna('').replace('없음', '').str.strip().ne('')).mean(),
    '두 소수 클래스 전체': sum(map(has_both, minority)) / len(minority),
    '924건 전체': sum(map(has_both, rows)) / len(rows),
}).round(3)
display(overlap.to_frame('blocker와 cost_basis를 둘 다 가진 비율'))
margin = (oof.filter(like='word_char_logistic_p_').apply(lambda row: row.nlargest(2).diff().abs().iloc[-1], axis=1))
display(margin.groupby(oof['gold'] == pred).describe().rename(index={True: '맞춘 건', False: '틀린 건'}).round(3))

## 읽는 법

- **오답의 3분의 1이 경계 하나에 있다.** 294건 중 98건이 `견적반영`과 `계약·질의검토`의
  상호 혼동입니다. `통상수용`만 F1 0.795이고 두 소수 클래스는 0.545, 0.573입니다.
- **합치는 방법은 결과를 바꾸지 않는다.** 2분류를 처음부터 학습한 값(0.7879)과 3분류
  예측을 사후에 접은 값(0.7876)이 사실상 같습니다. 3분류 모델이 이미 `통상수용` 대
  `검토필요` 축은 최대치로 가르고 있었다는 뜻입니다.
- **틀린 건은 모델이 망설인 건이다.** 1위·2위 확률차 중앙값이 오답 0.129, 정답 0.264로
  두 배 차이입니다. 오독이 아니라 경계에서 갈렸습니다.
- **경계 사례에는 겹침이 몰려 있다.** blocker와 원가 요인을 둘 다 가진 비율이 혼동 98건에서
  45.9%, 두 소수 클래스 전체에서 32.4%, 924건 전체에서 15.8%입니다. 결정 21은 겹칠 때
  blocker를 우선하지만 모델은 원가 신호를 따라갑니다.

따라서 3분류와 2분류의 차이 0.150은 표현이나 학습 절차가 아니라 **과제 정의에 속한
난이도**로 읽습니다. 개별 사례는 `reports/current/v4/boundary_cases.xlsx`에서 라벨 생성
당시의 `reasoning`과 함께 볼 수 있습니다.